# Step 2a: Train COOLANT (Simultaneous, Paper-faithful)

Train COOLANT with image-caption pairs using matched/unmatched pair construction.
- matched (caption_i, image_i) = Real (label 0)
- unmatched (caption_i, image_j) = Fake (label 1)
- **Balanced by construction** - no class weights needed

Prerequisites: Run `0_extract_pairs.ipynb` then `1_preprocess.ipynb` first.

Input: `./processed/coolant_{train,dev,test}.h5`
Output: `../../training/checkpoints_coolant/`

In [1]:
import sys
import os
import math
import random
import warnings
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")

project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [2]:
# Device and seeds
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

HDF5_DIR = Path("./processed")
SAVE_DIR = project_root / "training" / "checkpoints_coolant"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Save dir: {SAVE_DIR}")

Device: cuda
Save dir: g:\My Drive\Thesis_Final\fake-new-detection\training\checkpoints_coolant


In [3]:
# Load data
from src.processing.coolant import create_coolant_dataloaders

BATCH_SIZE = 32
IMAGE_DIM = 2048
TEXT_EMBED = 768

loaders, datasets = create_coolant_dataloaders(
    train_path=str(HDF5_DIR / "coolant_train.h5"),
    dev_path=str(HDF5_DIR / "coolant_dev.h5"),
    test_path=str(HDF5_DIR / "coolant_test.h5"),
    batch_size=BATCH_SIZE,
)

# Verify shapes
sample_cap, sample_img, sample_aid = datasets["train"][0]
print(f"\nCaption shape: {sample_cap.shape}")
print(f"Image shape: {sample_img.shape}")

CoolantPairDataset: 3537 pairs from coolant_train.h5
CoolantPairDataset: 1054 pairs from coolant_dev.h5
CoolantPairDataset: 876 pairs from coolant_test.h5

COOLANT DataLoaders created:
  Train: 111 batches (3537 pairs)
  Dev:   33 batches (1054 pairs)
  Test:  28 batches (876 pairs)

Caption shape: torch.Size([768, 128])
Image shape: torch.Size([2048])


In [4]:
# Config
CONFIG = {
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,
    "h_dim": 64,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "max_epochs": 30,
    "patience": 7,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
}
print("Config defined")

Config defined


In [5]:
# Create and patch model
from src.models.resnet_coolant import (
    ResNetCOOLANT, patch_encoding, patch_clip_projection, patch_cnn_with_dropout,
)

model = ResNetCOOLANT(CONFIG)

patch_encoding(model.similarity_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.ambiguity_module.encoding, image_dim=IMAGE_DIM)
patch_clip_projection(model.clip_module, target_dim=IMAGE_DIM, is_image=True)
patch_clip_projection(model.clip_module, target_dim=TEXT_EMBED, is_image=False)
patch_cnn_with_dropout(model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.ambiguity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready: {n_params:,} params")

Model ready: 5,610,288 params


In [6]:
# Optimizers and schedulers
optim_sim = torch.optim.AdamW(model.similarity_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
optim_clip = torch.optim.AdamW(model.clip_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
optim_det = torch.optim.AdamW(model.detection_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

steps_per_epoch = len(loaders["train"])
total_steps = steps_per_epoch * CONFIG["max_epochs"]

sch_sim = torch.optim.lr_scheduler.CosineAnnealingLR(optim_sim, T_max=CONFIG["max_epochs"])
sch_clip = torch.optim.lr_scheduler.CosineAnnealingLR(optim_clip, T_max=CONFIG["max_epochs"])
sch_det = torch.optim.lr_scheduler.CosineAnnealingLR(optim_det, T_max=CONFIG["max_epochs"])

print("Optimizers and schedulers created")

Optimizers and schedulers created


In [7]:
# Loss functions - NO class weights (balanced by construction)
from src.processing.coolant.training_utils import make_coolant_pairs, make_detection_batch, soft_cross_entropy

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce_det = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])  # For detection (2-class)
loss_ce_clip = nn.CrossEntropyLoss()  # For CLIP contrastive (B-class, no smoothing)
loss_kl = nn.KLDivLoss(reduction="batchmean")

print("Loss functions created (separate CE for detection vs CLIP)")

Loss functions created (separate CE for detection vs CLIP)


In [8]:
# Training loop
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss, tot_n = 0.0, 0
    all_y, all_p = [], []
    n_batches = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for caption, image, article_id in tqdm(loader, desc="Train" if train else "Eval"):
            caption = caption.to(DEVICE)
            image = image.to(DEVICE)
            bs = caption.size(0)
            if bs < 4:
                continue

            # Task 1a: Similarity (matched vs unmatched)
            cap_anchor, img_matched, img_unmatched = make_coolant_pairs(caption, image)
            ta_m, ia_m, _ = model.similarity_module(cap_anchor, img_matched)
            ta_u, ia_u, _ = model.similarity_module(cap_anchor, img_unmatched)
            t_cat = torch.cat([ta_m, ta_u])
            i_cat = torch.cat([ia_m, ia_u])
            y_cos = torch.cat([
                torch.ones(ta_m.size(0), device=DEVICE),
                -torch.ones(ta_u.size(0), device=DEVICE),
            ])
            ls = loss_cos(t_cat, i_cat, y_cos)

            if train:
                optim_sim.zero_grad()
                ls.backward()
                torch.nn.utils.clip_grad_norm_(model.similarity_module.parameters(), CONFIG["grad_clip"])
                optim_sim.step()

            # Task 1b: CLIP (unweighted CE for contrastive)
            cap_pooled = caption.mean(dim=2)
            ie, te = model.clip_module(image, cap_pooled)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)
            ts, is_, _ = model.similarity_module(caption, image)
            soft_m = is_ @ ts.T * math.exp(0.07)
            lc = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2
            ls2 = (soft_cross_entropy(logits, F.softmax(soft_m, 1)) + soft_cross_entropy(logits.T, F.softmax(soft_m.T, 1))) / 2
            l_clip = lc + 0.2 * ls2

            if train:
                optim_clip.zero_grad()
                l_clip.backward()
                torch.nn.utils.clip_grad_norm_(model.clip_module.parameters(), CONFIG["grad_clip"])
                optim_clip.step()

            # Task 2: Detection (label-smoothed CE for binary classification)
            det_cap, det_img, det_labels = make_detection_batch(caption, image)
            det_cap_pooled = det_cap.mean(dim=2)
            with torch.no_grad() if not train else torch.enable_grad():
                ie2, te2 = model.clip_module(det_img, det_cap_pooled)
            det_out, attn, skl = model.detection_module(det_cap, det_img, te2, ie2)
            ld = loss_ce_det(det_out, det_labels) + 0.5 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )

            if train:
                optim_det.zero_grad()
                ld.backward()
                torch.nn.utils.clip_grad_norm_(model.detection_module.parameters(), CONFIG["grad_clip"])
                optim_det.step()

            pred = det_out.argmax(1)
            tot_n += det_labels.size(0)
            tot_loss += ld.item() * det_labels.size(0)
            all_y.extend(det_labels.cpu().numpy())
            all_p.extend(pred.cpu().numpy())
            n_batches += 1

    macro_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)
    acc = accuracy_score(all_y, all_p)
    prec, rec, _, _ = precision_recall_fscore_support(all_y, all_p, average=None, zero_division=0)

    return {
        "loss": tot_loss / max(tot_n, 1),
        "acc": acc,
        "macro_f1": macro_f1,
        "real_prec": prec[0] if len(prec) > 0 else 0,
        "real_rec": rec[0] if len(rec) > 0 else 0,
        "fake_prec": prec[1] if len(prec) > 1 else 0,
        "fake_rec": rec[1] if len(rec) > 1 else 0,
    }, all_y, all_p

print("run_epoch defined (separate losses for CLIP vs detection)")

run_epoch defined (separate losses for CLIP vs detection)


In [9]:
# Training
best_val_f1 = 0.0
patience_counter = 0

print(f"Training for max {CONFIG['max_epochs']} epochs (patience={CONFIG['patience']})\n")

for epoch in range(1, CONFIG["max_epochs"] + 1):
    train_m, _, _ = run_epoch(loaders["train"], train=True)
    val_m, val_y, val_p = run_epoch(loaders["dev"], train=False)

    sch_sim.step()
    sch_clip.step()
    sch_det.step()

    cm = confusion_matrix(val_y, val_p)
    print(f"Epoch [{epoch:02d}/{CONFIG['max_epochs']}]")
    print(f"  Train: loss={train_m['loss']:.4f} acc={train_m['acc']:.4f} F1={train_m['macro_f1']:.4f}")
    print(f"  Val:   loss={val_m['loss']:.4f} acc={val_m['acc']:.4f} F1={val_m['macro_f1']:.4f}")
    print(f"  Val:   real_rec={val_m['real_rec']:.3f} fake_rec={val_m['fake_rec']:.3f}")
    print(f"  CM: {cm.tolist()}")

    if val_m["macro_f1"] > best_val_f1:
        best_val_f1 = val_m["macro_f1"]
        patience_counter = 0
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_f1": best_val_f1, "config": CONFIG}, SAVE_DIR / "best_model.pth")
        print(f"  >> Best F1={best_val_f1:.4f}, saved")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"\nEarly stopping at epoch {epoch}")
            break
    print()

print(f"\nDone. Best val F1: {best_val_f1:.4f}")

Training for max 30 epochs (patience=7)



Eval: 100%|██████████| 33/33 [00:34<00:00,  1.05s/it]


Epoch [01/30]
  Train: loss=0.7035 acc=0.5089 F1=0.5075
  Val:   loss=0.7002 acc=0.4948 F1=0.4901
  Val:   real_rec=0.398 fake_rec=0.591
  CM: [[420, 634], [431, 623]]
  >> Best F1=0.4901, saved



Eval: 100%|██████████| 33/33 [00:34<00:00,  1.05s/it]


Epoch [02/30]
  Train: loss=0.6956 acc=0.5123 F1=0.5122
  Val:   loss=0.6933 acc=0.5323 F1=0.4848
  Val:   real_rec=0.229 fake_rec=0.836
  CM: [[241, 813], [173, 881]]



...

Epoch [08/30]
  Train: loss=0.4774 acc=0.8202 F1=0.8193
  Val:   loss=1.3566 acc=0.5000 F1=0.3333
  Val:   real_rec=0.000 fake_rec=1.000
  CM: [[0, 1054], [0, 1054]]



Eval: 100%|██████████| 33/33 [00:37<00:00,  1.13s/it]


Epoch [09/30]
  Train: loss=0.4457 acc=0.8461 F1=0.8454
  Val:   loss=1.3746 acc=0.5024 F1=0.3443
  Val:   real_rec=0.011 fake_rec=0.993
  CM: [[12, 1042], [7, 1047]]



Eval: 100%|██████████| 33/33 [00:40<00:00,  1.22s/it]

Epoch [10/30]
  Train: loss=0.4245 acc=0.8632 F1=0.8626
  Val:   loss=1.3073 acc=0.5119 F1=0.3734
  Val:   real_rec=0.042 fake_rec=0.982
  CM: [[44, 1010], [19, 1035]]

Early stopping at epoch 10

Done. Best val F1: 0.5820


In [10]:
# Test evaluation
checkpoint = torch.load(SAVE_DIR / "best_model.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])

test_m, test_y, test_p = run_epoch(loaders["test"], train=False)
from sklearn.metrics import classification_report

print(f"Test: loss={test_m['loss']:.4f} acc={test_m['acc']:.4f} F1={test_m['macro_f1']:.4f}")
print(classification_report(test_y, test_p, target_names=["Real(matched)", "Fake(unmatched)"], digits=4))
print(f"Confusion Matrix: {confusion_matrix(test_y, test_p).tolist()}")

Eval: 100%|██████████| 28/28 [00:35<00:00,  1.26s/it]

Test: loss=0.6834 acc=0.5599 F1=0.5599
                 precision    recall  f1-score   support

  Real(matched)     0.5599    0.5605    0.5602       876
Fake(unmatched)     0.5600    0.5594    0.5597       876

       accuracy                         0.5599      1752
      macro avg     0.5599    0.5599    0.5599      1752
   weighted avg     0.5599    0.5599    0.5599      1752

Confusion Matrix: [[491, 385], [386, 490]]
